In [1]:
# -*- coding: utf-8 -*-
"""s3_dynamic_nli_consistency.py

S3 — Output-vs-output NLI consistency, RETRIEVER axis (E3), document-level.
Reads the per-example CSVs written by healthslm_eval_dynamic_fewshot.py
(one CSV per retriever) and computes, per test instance, bidirectional NLI
over the C(3,2)=3 unordered pairs of the 3 retriever outputs
(= 6 directed passes per instance). Same backbone, same composite formula
and thresholds as s3_nli_consistency.py, so E2 and E3 numbers are directly
comparable.

Input format notes (healthslm_eval_* per_example CSVs):
  - full generation text is in the 'extracted' column ('raw_prediction' is
    TRUNCATED to 200 chars — never use it for scoring)
  - there is NO instance_id: rows are in test-file order. Alignment across
    the three CSVs is positional WITHIN each dataset, verified by requiring
    the 'reference' strings to match at every position (abort on mismatch).

Per pair (A,B), with e = P(entailment), c = P(contradiction):
    composite  s(A,B) = 0.5*(e_fwd + e_bwd) - max(c_fwd, c_bwd)
    pair is "contradictory" if max(c_fwd, c_bwd) > CONTRA_THRESHOLD

Outputs (in OUT_DIR):
  s3retr_{NLI_SLUG}_{MODEL_SLUG}_per_pair.csv      — task x instance x retriever-pair
  s3retr_{NLI_SLUG}_{MODEL_SLUG}_per_instance.csv  — task x instance
  s3retr_{NLI_SLUG}_{MODEL_SLUG}_summary.csv       — task
  (column names match the random-draw S3 files -> E2 vs E3 is a plain concat)

Run:
    pip install transformers torch sentencepiece pandas
    python s3_dynamic_nli_consistency.py
"""

import os
import itertools
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------------------
# Config
# ------------------------------------------------------------------------

MODEL_NAME = "epfl-llm/meditron-7b"   # <- must match the eval runs
MODEL_SLUG = MODEL_NAME.split('/')[-1].lower().replace('-', '_').replace('.', '_')

OUT_DIR = '/workspace/demo_sensitivity_runs'

# Per-example CSVs from healthslm_eval_dynamic_fewshot.py (in /workspace/results).
# Filenames are inconsistent across runs (with/without 'pool100'), so resolve
# by glob. Keys are the retriever labels used in per_pair output.
import glob as _glob
RESULTS_DIR = '/workspace/results'

def _find_per_example(retriever_pat):
    pats = [f'{RESULTS_DIR}/results_dynamic_k5*{MODEL_SLUG}*{retriever_pat}*per_example*.csv']
    hits = sorted(set(sum((_glob.glob(p) for p in pats), [])))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly 1 per-example CSV for '{retriever_pat}', "
            f"found {len(hits)}: {hits}")
    return hits[0]

RETRIEVER_CSVS = {
    's_pubmedbert': _find_per_example('s_pubmedbert'),
    'bge':          _find_per_example('bge'),
    'tfidf':        _find_per_example('tfidf'),
}
print("Per-example CSVs:")
for r, p in RETRIEVER_CSVS.items():
    print(f"  {r}: {os.path.basename(p)}")

# Canonical task label -> 'dataset' value in the per-example CSVs (filenames).
# Canonical labels match the seed-axis S3 CSVs for the E2-vs-E3 join.
# aci excluded per design (pool/truncation degeneracy).
TASK_MAP = {
    'meddialog':      'meddialog_test_sample.jsonl',
    'medicationqa':   'MedicationQA_test_sample.jsonl',
    'mtsamples':      'mtsamples_test_sample.jsonl',
    'mtsamples_proc': 'mtsamples_procedures_test_sample.jsonl',
}
TASKS = list(TASK_MAP)

# Column holding the FULL generation text (see input format notes above)
TEXT_COL = 'extracted'
REF_COL = 'reference'
TASK_COL = 'dataset'

# Fixed NLI backbone for ALL models/axes (locked 2026-08-18). Never mix.
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
NLI_SLUG = NLI_MODEL.split('/')[-1].split('-mnli')[0].lower().replace('-', '_')
MAX_LENGTH = 512
BATCH_SIZE = 16
CONTRA_THRESHOLD = 0.5

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RETRIEVERS = list(RETRIEVER_CSVS)

# ------------------------------------------------------------------------
# Load + align the three per-example CSVs
# ------------------------------------------------------------------------

def load_retriever_outputs():
    """{task: {retriever: [output_text, ...]}} with positional alignment
    verified via the reference column."""
    frames = {}
    for r, path in RETRIEVER_CSVS.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"missing per-example CSV for '{r}': {path}")
        df = pd.read_csv(path, keep_default_na=False, dtype=str)
        assert TEXT_COL in df.columns, f"{path}: no '{TEXT_COL}' column"
        frames[r] = df
        print(f"  [{r}] columns: {df.columns.tolist()}")
        print(f"  [{r}] {TASK_COL} values: {df[TASK_COL].value_counts().to_dict()}")

    per_task = {}
    for task in TASKS:
        subs = {r: f[f[TASK_COL] == TASK_MAP[task]].reset_index(drop=True)
                for r, f in frames.items()}
        ns = {r: len(s) for r, s in subs.items()}
        if len(set(ns.values())) != 1 or 0 in ns.values():
            print(f"  [skip] {task}: row counts differ or empty: {ns}")
            continue
        # alignment guard: references must be identical at every position
        base = subs[RETRIEVERS[0]][REF_COL]
        for r in RETRIEVERS[1:]:
            mism = (subs[r][REF_COL] != base)
            if mism.any():
                raise AssertionError(
                    f"{task}: {int(mism.sum())} reference mismatches between "
                    f"'{RETRIEVERS[0]}' and '{r}' — row order differs, cannot "
                    f"align positionally. Fix before scoring.")
        per_task[task] = {r: subs[r][TEXT_COL].tolist() for r in RETRIEVERS}
    return per_task

# ------------------------------------------------------------------------
# NLI scorer (identical to s3_nli_consistency.py)
# ------------------------------------------------------------------------

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print(f"Loading NLI model {NLI_MODEL} on {DEVICE} ...")
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()

id2label = {i: l.lower() for i, l in nli.config.id2label.items()}
ENT_IDX = next(i for i, l in id2label.items() if 'entail' in l)
CON_IDX = next(i for i, l in id2label.items() if 'contra' in l)
print(f"  labels: {id2label}  (entail={ENT_IDX}, contra={CON_IDX})")


@torch.no_grad()
def nli_probs(premises, hypotheses):
    ents, cons = [], []
    for i in range(0, len(premises), BATCH_SIZE):
        enc = tokenizer(premises[i:i + BATCH_SIZE], hypotheses[i:i + BATCH_SIZE],
                        truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors='pt').to(DEVICE)
        probs = torch.softmax(nli(**enc).logits, dim=-1).cpu().numpy()
        ents.append(probs[:, ENT_IDX])
        cons.append(probs[:, CON_IDX])
    return np.concatenate(ents), np.concatenate(cons)

# ------------------------------------------------------------------------
# Score
# ------------------------------------------------------------------------

per_pair_rows, per_instance_rows = [], []
data = load_retriever_outputs()

for task, outs_by_r in data.items():
    n_inst = len(outs_by_r[RETRIEVERS[0]])
    print(f"\n=== {task} === {n_inst} instances")

    prem, hyp, owner = [], [], []
    inst_info = {}
    for iid in range(n_inst):
        outs = [(r, outs_by_r[r][iid]) for r in RETRIEVERS]
        valid = [(r, o) for r, o in outs if o and o.strip()]
        inst_info[iid] = {'n_valid': len(valid), 'n_empty': len(outs) - len(valid),
                          'pairs': list(itertools.combinations(valid, 2))}
        for p_idx, ((ra, a), (rb, b)) in enumerate(inst_info[iid]['pairs']):
            prem.append(a); hyp.append(b); owner.append((iid, p_idx, 'fwd'))
            prem.append(b); hyp.append(a); owner.append((iid, p_idx, 'bwd'))

    print(f"  scoring {len(prem)} directed pairs ...")
    E, C = nli_probs(prem, hyp) if prem else (np.array([]), np.array([]))

    directed = {}
    for (iid, p_idx, d), e, c in zip(owner, E, C):
        directed[(iid, p_idx, d)] = (float(e), float(c))

    for iid in range(n_inst):
        pairs_meta = inst_info[iid]['pairs']
        pairs = []
        for p_idx, ((ra, _), (rb, _)) in enumerate(pairs_meta):
            e_f, c_f = directed[(iid, p_idx, 'fwd')]
            e_b, c_b = directed[(iid, p_idx, 'bwd')]
            s = 0.5 * (e_f + e_b) - max(c_f, c_b)
            contra = max(c_f, c_b) > CONTRA_THRESHOLD
            pairs.append({'s': s, 'e': 0.5 * (e_f + e_b),
                          'c': max(c_f, c_b), 'contra': contra})
            per_pair_rows.append({
                'model': MODEL_NAME, 'task': task, 'instance_id': iid,
                'pair_idx': p_idx, 'retriever_a': ra, 'retriever_b': rb,
                'entail_fwd': e_f, 'entail_bwd': e_b,
                'contra_fwd': c_f, 'contra_bwd': c_b,
                'composite_s': s, 'is_contradiction': contra,
            })
        measurable = inst_info[iid]['n_valid'] >= 2
        per_instance_rows.append({
            'model': MODEL_NAME, 'task': task, 'instance_id': iid,
            'n_valid_outputs': inst_info[iid]['n_valid'],
            'n_empty_outputs': inst_info[iid]['n_empty'],
            'n_pairs': len(pairs),
            'nli_composite_s':    float(np.mean([p['s'] for p in pairs])) if measurable else np.nan,
            'mean_entailment':    float(np.mean([p['e'] for p in pairs])) if measurable else np.nan,
            'mean_contradiction': float(np.mean([p['c'] for p in pairs])) if measurable else np.nan,
            'frac_contra_pairs':  float(np.mean([p['contra'] for p in pairs])) if measurable else np.nan,
        })

# ------------------------------------------------------------------------
# Save (mirrors s3_nli_consistency.py; 's3retr_' prefix marks the axis)
# ------------------------------------------------------------------------

pd.DataFrame(per_pair_rows).to_csv(
    os.path.join(OUT_DIR, f's3retr_{NLI_SLUG}_{MODEL_SLUG}_per_pair.csv'), index=False)

if not per_instance_rows:
    raise SystemExit(
        "\n[ERROR] nothing scored — no task in TASKS matched the CSVs' "
        f"'{TASK_COL}' values (see the value_counts printed above). "
        "Update TASKS to the exact dataset names used in the per-example files.")

df = pd.DataFrame(per_instance_rows)
per_inst_csv = os.path.join(OUT_DIR, f's3retr_{NLI_SLUG}_{MODEL_SLUG}_per_instance.csv')
df.to_csv(per_inst_csv, index=False)

summary = (df.groupby('task')
             .agg(n_instances=('instance_id', 'count'),
                  n_low_valid=('n_valid_outputs', lambda s: int((s < 2).sum())),
                  mean_composite_s=('nli_composite_s', 'mean'),
                  median_composite_s=('nli_composite_s', 'median'),
                  mean_entailment=('mean_entailment', 'mean'),
                  mean_contradiction=('mean_contradiction', 'mean'),
                  frac_contra_pairs=('frac_contra_pairs', 'mean'),
                  p10_composite_s=('nli_composite_s', lambda s: s.quantile(0.10)),
                  p90_composite_s=('nli_composite_s', lambda s: s.quantile(0.90)))
             .round(4)
             .reset_index())
summary.insert(0, 'model', MODEL_NAME)
summary['axis'] = 'retriever'
summary['nli_model'] = NLI_MODEL
summary['contra_threshold'] = CONTRA_THRESHOLD

summary_csv = os.path.join(OUT_DIR, f's3retr_{NLI_SLUG}_{MODEL_SLUG}_summary.csv')
summary.to_csv(summary_csv, index=False)

print(f"\n### S3 retriever-axis summary — {MODEL_NAME} ###")
print(summary.to_string(index=False))
print(f"\nSaved -> {per_inst_csv}")
print(f"      -> {summary_csv}")


Per-example CSVs:
  s_pubmedbert: results_dynamic_k5_pool100_meditron_7b_s_pubmedbert_ms_marco_per_example.csv
  bge: results_dynamic_k5_pool100_meditron_7b_bge_base_en_v1_5_per_example.csv
  tfidf: results_dynamic_k5_pool100_meditron_7b_tfidf_per_example.csv
Loading NLI model MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli on cuda ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

  labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}  (entail=0, contra=2)
  [s_pubmedbert] columns: ['dataset', 'task', 'k_shot', 'reference', 'bounds', 'raw_prediction', 'extracted', 'retrieval_sim']
  [s_pubmedbert] dataset values: {'meddialog_test_sample.jsonl': 500, 'MedicationQA_test_sample.jsonl': 500, 'mtsamples_test_sample.jsonl': 500, 'mtsamples_procedures_test_sample.jsonl': 500}
  [bge] columns: ['dataset', 'task', 'k_shot', 'reference', 'bounds', 'raw_prediction', 'extracted', 'retrieval_sim']
  [bge] dataset values: {'meddialog_test_sample.jsonl': 500, 'MedicationQA_test_sample.jsonl': 500, 'mtsamples_test_sample.jsonl': 500, 'mtsamples_procedures_test_sample.jsonl': 500}
  [tfidf] columns: ['dataset', 'task', 'k_shot', 'reference', 'bounds', 'raw_prediction', 'extracted', 'retrieval_sim']
  [tfidf] dataset values: {'meddialog_test_sample.jsonl': 500, 'MedicationQA_test_sample.jsonl': 500, 'mtsamples_test_sample.jsonl': 500, 'mtsamples_procedures_test_sample.json